# Qwen3.5-4B atomic-supervision sweep

This notebook audits the fixed BFCL and CommitPack atomic datasets and analyzes the staged LoRA sweep. It is safe to run while jobs are incomplete: missing cells are reported rather than treated as zero accuracy. Hyperparameters are selected on validation data; test results appear only in the final section.

In [1]:
from __future__ import annotations
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'self').exists():
    repo_root = repo_root.parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
configured = os.environ.get('CODING_SWEEP_ROOT')
if configured:
    run_root = Path(configured).expanduser().resolve()
else:
    candidates = sorted((repo_root / 'artifacts/runs').glob('coding_atomic_sweep_*'))
    if not candidates:
        raise FileNotFoundError('Set CODING_SWEEP_ROOT or prepare a coding_atomic_sweep run.')
    run_root = candidates[-1]
data_dir = run_root / 'data'
print('Run root:', run_root)

Run root: /scratch/gpfs/BRENDEN/changho/compositional-something/artifacts/runs/coding_atomic_sweep_20260718_014707


## Settings, representative records, and guard-rule audit

This section makes the experimental contract concrete: it shows the staged grid, one record from every supervised/evaluation split, strict source-data rejection counts, and evaluator behavior on gold, malformed, and behaviorally wrong outputs.

In [2]:
from self.coding.atomic_data import parse_config_document, read_examples
from self.coding.evaluation import evaluate_example
LEARNING_RATES = (1e-5, 5e-5, 2e-4)
TASK_SETTINGS = {
    'bfcl': {'data_sizes': (30, 60, 120, 240), 'steps': (10, 30, 100), 'max_length': 512, 'micro_batch_size': 16},
    'commitpack': {'data_sizes': (250, 500, 1000, 2000), 'steps': (50, 150, 450), 'max_length': 1024, 'micro_batch_size': 4},
}

settings = []
for task, cfg in TASK_SETTINGS.items():
    settings.append({
        'task': task, 'data sizes': cfg['data_sizes'], 'optimizer steps': cfg['steps'],
        'learning rates': LEARNING_RATES, 'sequence limit': cfg['max_length'],
        'microbatch': cfg['micro_batch_size'], 'effective batch': 16,
        'adapter': 'LoRA r=16, alpha=32, all-linear',
    })
display(pd.DataFrame(settings))

audits = {}
for task in ('bfcl', 'commitpack'):
    path = data_dir / task / 'audit.json'
    audits[task] = json.loads(path.read_text())
display(pd.DataFrame([
    {'task': task, **audit.get('split_counts', audit.get('selected_split_counts', {}))}
    for task, audit in audits.items()
]))
if 'global_stratum_counts' in audits['commitpack']:
    display(pd.Series(audits['commitpack']['global_stratum_counts'], name='eligible atoms').to_frame())
rejections = pd.Series(audits['commitpack'].get('rejection_counts', {}), name='rows').sort_values(ascending=False)
display(rejections.head(20).to_frame())

samples, preview_rows = {}, []
for task in ('bfcl', 'commitpack'):
    for split in ('train', 'validation', 'test'):
        example = read_examples(data_dir / task / f'{split}.jsonl')[0]
        samples[(task, split)] = example
        user_text = example.messages[-1]['content']
        preview_rows.append({
            'task': task, 'split': split, 'source id': example.source_id,
            'components': example.component_count,
            'user prompt': user_text[:700] + (' ...' if len(user_text) > 700 else ''),
            'target': example.target,
        })
display(pd.DataFrame(preview_rows))

guard_rows = []
for task in ('bfcl', 'commitpack'):
    example = samples[(task, 'validation')]
    cases = {
        'gold': example.target,
        'markdown fence': f'```json\n{example.target}\n```',
        'empty but valid array': '[]',
        'invalid behavior': ('[{"name":"unknown","arguments":{}}]' if task == 'bfcl'
                             else '[{"op":"remove","path":"/definitely_missing"}]'),
    }
    for case, prediction in cases.items():
        result = evaluate_example(example, prediction)
        guard_rows.append({
            'task': task, 'case': case, 'format valid': result.format_valid,
            'behavior valid': result.behavior_valid, 'exact': result.exact, 'error': result.error,
        })
display(pd.DataFrame(guard_rows))

yaml_guards = {
    'duplicate key': 'a: 1\na: 2\n',
    'anchor/alias': 'a: &v 1\nb: *v\n',
    'merge key': 'base: {a: 1}\nderived: {<<: {a: 1}}\n',
    'non-JSON timestamp': 'when: 2026-07-18\n',
}
yaml_guard_rows = []
for case, document in yaml_guards.items():
    try:
        parse_config_document(document, 'yaml')
        outcome = 'INCORRECTLY ACCEPTED'
    except Exception as exc:
        outcome = f'rejected: {type(exc).__name__}: {exc}'
    yaml_guard_rows.append({'case': case, 'outcome': outcome})
display(pd.DataFrame(yaml_guard_rows))

,task,data sizes,optimizer steps,learning rates,sequence limit,microbatch,effective batch,adapter
0,bfcl,"(30, 60, 120, 240)","(10, 30, 100)","(1e-05, 5e-05, 0.0002)",512,16,16,"LoRA r=16, alpha=32, all-linear"
1,commitpack,"(250, 500, 1000, 2000)","(50, 150, 450)","(1e-05, 5e-05, 0.0002)",1024,4,16,"LoRA r=16, alpha=32, all-linear"


,task,hidden_composition,test,train,validation,frontier_2,frontier_4,frontier_8
0,bfcl,60.0,60,240,40,NaN,NaN,NaN
1,commitpack,NaN,1200,2000,600,200.0,74.0,11.0


,eligible atoms
json:add,4987
json:remove,1789
json:replace,20036
yaml:add,3804
yaml:remove,1043
yaml:replace,8002


,rows
array_change,56977
container_add,14033
yaml_multiple_documents,10933
yaml_non_string_key,9960
container_remove,6614
yaml_custom_tag,4415
json_parse_error,3222
yaml_parse_error,3178
container_type_change,1323
no_supported_change,990


,task,split,source id,components,user prompt,target
0,bfcl,train,simple_18,1,User request:\nFind the prime factors of the n...,"[{""arguments"":{""number"":123456},""name"":""number..."
1,bfcl,validation,simple_272,1,User request:\nCalculate the area and circumfe...,"[{""arguments"":{""radius"":5},""name"":""calculate_c..."
2,bfcl,test,simple_97,1,User request:\nCalculate the factorial of the ...,"[{""arguments"":{""number"":5},""name"":""math.factor..."
3,commitpack,train,commitpack-5059ed5256739d0b4949,1,Configuration language: JSON\nContext rooted a...,"[{""op"":""add"",""path"":""/require-dev/phpdocumento..."
4,commitpack,validation,commitpack-37fd06e357ae18e3d842,1,Configuration language: JSON\nContext rooted a...,"[{""op"":""add"",""path"":""/dependencies/less"",""valu..."
5,commitpack,test,commitpack-5d86304af072a7bd70e6,1,Configuration language: JSON\nContext rooted a...,"[{""op"":""add"",""path"":""/scripts/build"",""value"":""..."


,task,case,format valid,behavior valid,exact,error
0,bfcl,gold,True,True,True,None
1,bfcl,markdown fence,False,False,False,response is not a bare JSON array
2,bfcl,empty but valid array,True,True,False,accepted-set mismatch
3,bfcl,invalid behavior,True,False,False,schema mismatch
4,commitpack,gold,True,True,True,None
5,commitpack,markdown fence,False,False,False,response is not a bare JSON array
6,commitpack,empty but valid array,True,True,False,patched document differs from intended state
7,commitpack,invalid behavior,True,False,False,Remove target is missing: /definitely_missing


,case,outcome
0,duplicate key,rejected: ValueError: Duplicate YAML mapping k...
1,anchor/alias,rejected: ValueError: YAML aliases and anchors...
2,merge key,rejected: ValueError: YAML merge keys are not ...
3,non-JSON timestamp,rejected: ValueError: Non-JSON value of type d...


## Zero-shot baselines, smoke gates, and completed validation cells

In [3]:
baseline_rows = []
for task in ('bfcl', 'commitpack'):
    path = run_root / 'base' / task / 'metrics.json'
    if path.exists():
        payload = json.loads(path.read_text())
        for split, metrics in payload.items():
            if isinstance(metrics, dict) and 'exact_accuracy' in metrics:
                baseline_rows.append({'task': task, 'split': split, **metrics})
display(pd.DataFrame(baseline_rows))

smoke_rows = []
for task in ('bfcl', 'commitpack'):
    path = run_root / 'smoke' / task / 'metrics.json'
    if path.exists():
        payload = json.loads(path.read_text())
        smoke_rows.append({
            'task': task, 'data size': payload['cell']['data_size'], 'steps': payload['cell']['max_steps'],
            'learning rate': payload['cell']['learning_rate'],
            'validation exact': payload['validation']['exact_accuracy'],
            'validation format': payload['validation']['format_accuracy'],
            'microbatch': payload['training']['micro_batch_size'],
            'peak GPU GiB': payload['peak_cuda_memory_bytes'] / 2**30,
        })
display(pd.DataFrame(smoke_rows))

rows = []
for path in sorted((run_root / 'cells').glob('*/*/metrics.json')):
    payload = json.loads(path.read_text())
    cell, val = payload['cell'], payload['validation']
    rows.append({
        **cell,
        'validation_exact': val['exact_accuracy'],
        'validation_format': val['format_accuracy'],
        'train_loss': payload['training'].get('train_loss'),
        'runtime_seconds': payload['training'].get('train_runtime'),
        'peak_gpu_gib': payload.get('peak_cuda_memory_bytes', 0) / 2**30,
    })
cells = pd.DataFrame(rows)
print(f'Completed {len(cells)} / 42 training cells')
display(cells.sort_values(['task', 'validation_exact'], ascending=[True, False]).head(20) if len(cells) else cells)

,task,split,behavior_valid_accuracy,count,error_counts,exact_accuracy,format_accuracy
0,bfcl,controlled_2,0.000000,200,{'response is not a bare JSON array': 200},0.000000,0.000000
1,bfcl,controlled_4,0.000000,200,{'response is not a bare JSON array': 200},0.000000,0.000000
2,bfcl,controlled_8,0.000000,200,{'response is not a bare JSON array': 200},0.000000,0.000000
3,bfcl,natural_parallel,0.005000,200,"{'accepted-set mismatch': 1, 'response is not ...",0.000000,0.005000
4,bfcl,natural_parallel_multiple,0.020000,200,"{'accepted-set mismatch': 1, 'response is not ...",0.015000,0.020000
5,bfcl,test,0.000000,60,{'response is not a bare JSON array': 60},0.000000,0.000000
6,bfcl,validation,0.000000,40,{'response is not a bare JSON array': 40},0.000000,0.000000
7,commitpack,natural_2,0.810000,200,"{'Patch path does not exist: /app/theme': 1, '...",0.775000,0.995000
8,commitpack,natural_4,0.783784,74,{'Patch path does not exist: /require-dev/php-...,0.770270,1.000000
9,commitpack,natural_8,0.909091,11,{'Patch path does not exist: /easyengine/maria...,0.909091,1.000000


,task,data size,steps,learning rate,validation exact,validation format,microbatch,peak GPU GiB
0,bfcl,30,1,0.00005,0.775000,1.000000,16,119.296904
1,commitpack,250,1,0.00005,0.848333,0.996667,4,44.688628


Completed 0 / 42 training cells


""


In [4]:
if len(cells):
    for task in ('bfcl', 'commitpack'):
        full_size = {'bfcl': 240, 'commitpack': 2000}[task]
        frame = cells[(cells.task == task) & (cells.seed == 7) & (cells.data_size == full_size)]
        if frame.empty:
            continue
        pivot = frame.pivot(index='learning_rate', columns='max_steps', values='validation_exact')
        fig, ax = plt.subplots(figsize=(6, 3))
        image = ax.imshow(pivot.values, vmin=0, vmax=1, cmap='viridis', aspect='auto')
        ax.set(xticks=range(len(pivot.columns)), xticklabels=pivot.columns, yticks=range(len(pivot.index)), yticklabels=pivot.index, xlabel='optimizer steps', ylabel='learning rate', title=f'{task}: validation exact accuracy')
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                value = pivot.iloc[i, j]
                if pd.notna(value): ax.text(j, i, f'{value:.2f}', ha='center', va='center', color='white')
        fig.colorbar(image, ax=ax); plt.show()

## Data scaling, replication stability, and efficiency

In [5]:
if len(cells):
    seed7 = cells[cells.seed == 7]
    for task, frame in seed7.groupby('task'):
        fig, ax = plt.subplots(figsize=(6, 3))
        for (steps, lr), curve in frame.groupby(['max_steps', 'learning_rate']):
            if curve.data_size.nunique() > 1:
                curve = curve.sort_values('data_size')
                ax.plot(curve.data_size, curve.validation_exact, marker='o', label=f'{steps} steps, lr={lr:g}')
        ax.set(xlabel='labeled atomic examples', ylabel='validation exact accuracy', ylim=(0, 1), title=f'{task}: data scaling')
        ax.legend(fontsize=8); plt.show()
    replicated = cells.groupby(['task','data_size','max_steps','learning_rate']).agg(mean_validation=('validation_exact','mean'), std_validation=('validation_exact','std'), seeds=('seed','nunique'), mean_runtime=('runtime_seconds','mean')).reset_index()
    display(replicated[replicated.seeds == 3].sort_values(['task','mean_validation'], ascending=[True,False]))

## Final held-out decision (not used for selection)

In [6]:
summary_path = run_root / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.DataFrame([{'task': task, **values} for task, values in summary['tasks'].items()]).set_index('task'))
else:
    print('Final selection/test evaluation is not complete yet.')

Final selection/test evaluation is not complete yet.
